In [ ]:
@debug_variables
def process_video_folder():
    
    INPUT_VIDEO_FOLDER_PATH = os.getenv("INPUT_VIDEO_FOLDER_PATH")
    if not os.path.exists(INPUT_VIDEO_FOLDER_PATH.replace("input_video", "output_video")):
        os.makedirs(INPUT_VIDEO_FOLDER_PATH.replace("input_video", "output_video"))
    for filename in os.listdir(INPUT_VIDEO_FOLDER_PATH):
        if ("small_test" in filename): #-fc-bayern-münchen-vs-bayer-04-leverkusen-24 -fc-bayern-münchen-vs-bayer-04-leverkusen-43 Denied One-on-One.mp4 -fc-bayern-münchen-vs-bayer-04-leverkusen-58 # rijul_test_case_viedo_1
            print(filename)
            logger.info(f"Processing video: {filename}")
            input_path = os.path.join(INPUT_VIDEO_FOLDER_PATH, filename)
            output_path = os.path.join(INPUT_VIDEO_FOLDER_PATH.replace("input_video", "output_video"), filename.split(".")[0]+"_annotated.mp4")
            
            debug_path = f"{os.getenv('DEBUG_PATH')}"
            stub_path = f"{os.getenv('STUBS')}"
            try:
                run_video(input_path, output_path, debug_path, stub_path)
            except Exception as e:
                print(str(e))
                traceback.print_exc()

### Decorators
- memroy_usage, time it, profiler, debug_variables (printing all the function variables for debugging)
- conditional decorators, basis on flag provided in the argument
- Others: Validate data type, 

In [ ]:
# Pandera decorators for DataFrame validation
import pandera as pa
from pandera.typing import DataFrame, Series

class InputSchema(pa.DataFrameModel):
    """Define expected data schema"""
    user_id: Series[int] = pa.Field(ge=0)
    age: Series[int] = pa.Field(ge=0, le=120)
    income: Series[float] = pa.Field(ge=0)
    score: Series[float] = pa.Field(ge=0, le=1)

class OutputSchema(pa.DataFrameModel):
    """Define output schema"""
    user_id: Series[int] = pa.Field(ge=0)
    prediction: Series[float] = pa.Field(ge=0, le=1)
    confidence: Series[float] = pa.Field(ge=0, le=1)

@pa.check_input(InputSchema, 0)  # Validate first argument
@pa.check_output(OutputSchema)   # Validate return value
def predict_user_scores(data: DataFrame[InputSchema]) -> DataFrame[OutputSchema]:
    """ML prediction with automatic I/O validation"""
    # Your prediction logic here
    predictions = model.predict_proba(data)[:, 1]
    return pd.DataFrame({
        'user_id': data['user_id'],
        'prediction': predictions,
        'confidence': np.abs(predictions - 0.5) * 2
    })

In [ ]:
def conditional_debug(env_var: str = "DEBUG_MODE"):
    """Decorator that only runs debug code if environment variable is set"""
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if os.getenv(env_var, "false").lower() in ("true", "1", "yes"):
                # Run with debug
                return comprehensive_debug(func)(*args, **kwargs)
            else:
                # Run without debug
                return func(*args, **kwargs)
        return wrapper
    return decorator

# Usage: Only debug when DEBUG_MODE=1
@conditional_debug("DEBUG_MODE")

In [ ]:
# Types of decorators for prduction ready code: Circuit Breakers, Smart retry mechanism, rate limiting, smart_cache, debug_logger, 
#Context managers handles resources automatically, thus garbage collection is handled well.
import contextlib

@contextlib.contextmanager
def debug_context(debug_controller, *features):
    """Context manager for debug control"""
    try:
        debug_controller.enable(*features)
        yield 
    finally:
        debug_controller.disable(*features)

def main():
    process_video_folder()
    # process_video_folderwise()

if __name__ == '__main__': 
    from config import local_settings
    if settings.IS_LOCAL:
        debug_controller = DebugController()
        
        # Use context manager for cleaner control
        with debug_context(debug_controller, "timing"):
            main()

In [ ]:
# Global debug controller
class DebugController:
    def __init__(self):
        self.enabled_features = set()
    
    def enable(self, *features):
        self.enabled_features.update(features)
    
    def disable(self, *features):
        self.enabled_features.difference_update(features)
    
    def is_enabled(self, feature: str) -> bool:
        return feature in self.enabled_features

# @timing_decorator #Make if else that runs only when debug_controller has timing enabled.
# This method is better compared to setting flag based on global config, which will always run timing whenever process_video_folder_test will be called
# But to make changes such that it don't run in one call and run in another function call then we can use this method of debugcontroller.
def process_video_folder_test():
    print("Processing video folder")
debug_controller = DebugController()
debug_controller.enable("timing")
process_video_folder_test()
debug_controller.disable("timing")


process_video_folder()
debug_controller.disable("timing")

In [ ]:
# MLflow experiment tracking
import mlflow
from mlflow import log_metric, log_param, log_artifact

def mlflow_experiment(experiment_name: str):
    """Decorator for automatic MLflow experiment tracking"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            mlflow.set_experiment(experiment_name)
            with mlflow.start_run():
                # Log function parameters
                sig = inspect.signature(func)
                bound_args = sig.bind(*args, **kwargs)
                bound_args.apply_defaults()
                
                for param_name, param_value in bound_args.arguments.items():
                    if isinstance(param_value, (int, float, str)):
                        log_param(param_name, param_value)
                
                result = func(*args, **kwargs)
                
                # Log results if they contain metrics
                if isinstance(result, dict):
                    for key, value in result.items():
                        if isinstance(value, (int, float)):
                            log_metric(key, value)
                
                return result
        return wrapper
    return decorator

@mlflow_experiment("user_churn_prediction")
def train_churn_model(data, learning_rate=0.01, max_depth=5):
    """Training with automatic experiment tracking"""
    model = XGBClassifier(learning_rate=learning_rate, max_depth=max_depth)
    X_train, X_test, y_train, y_test = train_test_split(data.drop('churn', axis=1), data['churn'])
    
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)
    
    return {'accuracy': accuracy, 'model': model}